In [1]:
!pip install pandas numpy torch tqdm transformers scikit-learn sentence-transformers scipy joblib

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.3 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 35.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 51.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 52.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 6.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 53.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import torch
from torch.optim import AdamW
from tqdm import tqdm
import copy
import json
from scipy.sparse import save_npz
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import hstack, csr_matrix
from tqdm import tqdm
import copy
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from scipy import sparse
import joblib

/home/ucloud/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# load preprocessed data
preprocessed_df = pd.read_csv('final_dataset_preprocessed_final.csv')

In [ ]:
# create directory to save features
save_dir = "saved_features_final"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# shape of the preprocessed dataset
print("Shape of the preprocessed dataset:", preprocessed_df.shape)

Shape of the preprocessed dataset: (40301, 51)


In [6]:
# print columns
preprocessed_df.columns

Index(['blurb', 'category_name', 'category_parent_name', 'country',
       'creator_has_admin_message_badge', 'creator_ppo_has_action',
       'creator_backing_action_count', 'currency', 'location_type', 'name',
       'photo_1536x864', 'prelaunch_activated', 'video',
       'campaign_duration_days', 'days_between_created_launched',
       'launched_day', 'launched_day_of_week', 'launched_hour',
       'launched_month', 'goal_converted_USD', 'goal_converted_USD_log',
       'success', 'num_reward_tiers', 'lowest_pledge_level_usd',
       'highest_pledge_level_usd', 'has_reward_images',
       'average_number_of_words_per_reward', 'has_restricted_shipping',
       'rewards_html_text', 'rewards_char_length', 'risks_html_text',
       'risks_char_length', 'total_media_assets', 'num_static_images',
       'num_animated_content', 'story_html_text', 'story_char_length',
       'blurb_char_length', 'ratio_text_to_total_assets', 'total_text_length',
       'is_local', 'blurb_nlp', 'name_nlp', 

In [ ]:
# convert boolean columns to integers
bool_cols = ['creator_has_admin_message_badge',
    'creator_ppo_has_action',
    'prelaunch_activated',
    'video',
    'has_reward_images',
    'has_restricted_shipping'
] 
preprocessed_df[bool_cols] = preprocessed_df[bool_cols].astype(int)
print(preprocessed_df[bool_cols].head())

   creator_has_admin_message_badge  creator_ppo_has_action  \
0                                0                       0   
1                                0                       0   
2                                0                       0   
3                                0                       0   
4                                0                       0   

   prelaunch_activated  video  has_reward_images  has_restricted_shipping  
0                    0      1                  0                        1  
1                    1      1                  1                        1  
2                    1      1                  0                        0  
3                    0      1                  0                        1  
4                    1      1                  1                        0  


In [ ]:
# define metadata features
metadata_features = ['category_parent_name', 'country',
       'creator_has_admin_message_badge', 'creator_ppo_has_action',
       'creator_backing_action_count', 'currency', 'location_type',
       'prelaunch_activated', 'video',
       'campaign_duration_days', 'days_between_created_launched',
       'launched_day', 'launched_day_of_week', 'launched_hour',
       'launched_month', 'goal_converted_USD_log',
       'num_reward_tiers', 'lowest_pledge_level_usd',
       'highest_pledge_level_usd', 'has_reward_images',
       'average_number_of_words_per_reward', 'has_restricted_shipping',
       'rewards_char_length', 'risks_char_length', 'total_media_assets', 'num_static_images',
       'num_animated_content', 'story_char_length',
       'ratio_text_to_total_assets', 'blurb_char_length', 'total_text_length',
       'is_local'
]

In [ ]:
# define textual features
textual_features_nlp = [
    'blurb_nlp',
    'name_nlp',
    'story_html_text_nlp',
    'risks_html_text_nlp',
    'rewards_html_text_nlp'
]

textual_features_llm = [
    'blurb_llm',
    'name_llm',
    'story_html_text_llm',
    'risks_html_text_llm',
    'rewards_html_text_llm'
]

In [ ]:
# split the data
def split_once(df, target="success", test_size=0.15, val_size=0.15, random_state=42):
    y = df[target]
    all_idx = df.index

    temp_size = test_size + val_size

    idx_train, idx_temp = train_test_split(
        all_idx,
        test_size=temp_size,
        random_state=random_state,
        stratify=y
    )

    y_temp = y.loc[idx_temp]
    test_fraction_of_temp = test_size / temp_size

    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=test_fraction_of_temp,
        random_state=random_state,
        stratify=y_temp
    )

    splits = {
        "train_idx": idx_train,
        "val_idx": idx_val,
        "test_idx": idx_test,
        "y_train": y.loc[idx_train],
        "y_val": y.loc[idx_val],
        "y_test": y.loc[idx_test],
    }
    return splits

# function to build X and y for one setting from the same precomputed split indices
def build_setting_data(df, splits, feature_cols):
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in df: {missing}")

    X_train = df.loc[splits["train_idx"], feature_cols].copy()
    X_val   = df.loc[splits["val_idx"], feature_cols].copy()
    X_test  = df.loc[splits["test_idx"], feature_cols].copy()

    y_train = splits["y_train"].copy()
    y_val   = splits["y_val"].copy()
    y_test  = splits["y_test"].copy()

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
# compute splits once and reuse across all experiments
splits = split_once(preprocessed_df, target="success")

In [ ]:
# extract indices and y splits for reuse
train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

In [ ]:
# save indices for reuse
np.save(os.path.join(save_dir, "train_idx.npy"), train_idx)
np.save(os.path.join(save_dir, "val_idx.npy"), val_idx)
np.save(os.path.join(save_dir, "test_idx.npy"), test_idx)

In [13]:
# metadata only setting
metadata_cols = metadata_features
X_train_meta, X_val_meta, X_test_meta, y_train_meta, y_val_meta, y_test_meta = build_setting_data(preprocessed_df, splits, metadata_cols)

In [ ]:
# combine textual features for NLP setting
preprocessed_df['combined_text_nlp'] = (
    preprocessed_df[textual_features_nlp]
    .fillna('')
    .agg(' '.join, axis=1)
)

# build text splits for NLP setting
X_train_nlp, X_val_nlp, X_test_nlp, y_train_nlp, y_val_nlp, y_test_nlp = build_setting_data(preprocessed_df, splits, ['combined_text_nlp'])

In [ ]:
# combine textual features for LLM setting
preprocessed_df['combined_text_llm'] = (
    preprocessed_df[textual_features_llm]
    .fillna('')
    .agg(' '.join, axis=1)
)

# build text splits for LLM setting
X_train_llm, X_val_llm, X_test_llm, y_train_llm, y_val_llm, y_test_llm = build_setting_data(preprocessed_df, splits, ['combined_text_llm'])

In [16]:
# Extract as Series for vectorizers
X_train_nlp = X_train_nlp['combined_text_nlp']
X_val_nlp = X_val_nlp['combined_text_nlp']
X_test_nlp = X_test_nlp['combined_text_nlp']

X_train_llm = X_train_llm['combined_text_llm']
X_val_llm = X_val_llm['combined_text_llm']
X_test_llm = X_test_llm['combined_text_llm']

print(f"X_train_nlp: {X_train_nlp.shape}")
print(f"X_val_nlp: {X_val_nlp.shape}")
print(f"X_test_nlp: {X_test_nlp.shape}")

X_train_nlp: (28210,)
X_val_nlp: (6045,)
X_test_nlp: (6046,)


Encoding

In [ ]:
# define categorical columns
categorical_columns = ['country', 'currency',
    'location_type',
    'category_parent_name'
    ]

In [ ]:
# one-hot encode categorical features
def one_hot_encode(X_train, X_val, X_test, columns):
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    # Fit only on training data
    encoder.fit(X_train[columns])
    feature_names = encoder.get_feature_names_out(columns)

    def transform(df):
        encoded = encoder.transform(df[columns])

        encoded_df = pd.DataFrame(
            encoded,
            columns=feature_names,
            index=df.index
        )

        df = df.drop(columns=columns)
        df = pd.concat([df, encoded_df], axis=1)
        df = df.reindex(sorted(df.columns), axis=1)

        return df

    return transform(X_train.copy()), transform(X_val.copy()), transform(X_test.copy()), encoder

In [19]:
# apply one-hot encoding to a copy of the preprocessed_df
X_train_meta_encoded, X_val_meta_encoded, X_test_meta_encoded, encoder = one_hot_encode(X_train_meta, X_val_meta, X_test_meta, categorical_columns)

print("Shape of the encoded dataset:", X_train_meta_encoded.shape)
print(f"Sample of one-hot encoded columns:\n{X_train_meta_encoded[encoder.get_feature_names_out(categorical_columns)].head()}")
print(f"Sample of One-Hot Encoded columns (first 5): {X_train_meta_encoded.filter(like='category_parent_name_').columns.tolist()[:5]}")

Shape of the encoded dataset: (28210, 91)
Sample of one-hot encoded columns:
       country_AT  country_AU  country_BE  country_CA  country_CH  country_DE  \
33459         0.0         0.0         0.0         0.0         0.0         0.0   
15275         0.0         0.0         0.0         0.0         0.0         0.0   
2661          0.0         0.0         0.0         0.0         0.0         0.0   
39442         0.0         0.0         0.0         0.0         0.0         0.0   
5207          0.0         0.0         0.0         0.0         0.0         0.0   

       country_DK  country_ES  country_FR  country_GB  ...  \
33459         0.0         0.0         0.0         0.0  ...   
15275         0.0         0.0         0.0         0.0  ...   
2661          0.0         0.0         0.0         0.0  ...   
39442         0.0         0.0         0.0         0.0  ...   
5207          0.0         0.0         0.0         0.0  ...   

       category_parent_name_Fashion  category_parent_name_Film 

In [ ]:
# print the new column names after one-hot encoding
one_hot_columns = list(
    encoder.get_feature_names_out(categorical_columns)
)
print(f"One-hot encoded columns: {one_hot_columns[:10]} ...")

One-hot encoded columns: ['country_AT', 'country_AU', 'country_BE', 'country_CA', 'country_CH', 'country_DE', 'country_DK', 'country_ES', 'country_FR', 'country_GB'] ...


In [ ]:
# define binary features
binary_features = [
    'creator_has_admin_message_badge',
    'creator_ppo_has_action',
    'prelaunch_activated',
    'video',
    'has_reward_images',
    'has_restricted_shipping'
]

Scaling

In [ ]:
# function to scale continuous features while keeping binary and one-hot encoded features unchanged
def scale_metadata_features(X_train, X_val, X_test, binary_features, one_hot_columns):
    all_features = X_train.columns.tolist()

    binary_features = [f for f in binary_features if f in all_features]
    one_hot_columns = [f for f in one_hot_columns if f in all_features]
    continuous_features = [f for f in all_features if f not in binary_features and f not in one_hot_columns]

    print(f"Binary features ({len(binary_features)}): {binary_features}")
    print(f"Continuous features ({len(continuous_features)}): {continuous_features}")

    scaler = StandardScaler()

    # Fit scaler only on training continuous features
    scaler.fit(X_train[continuous_features])

    def scale_df(df):
        df = df.copy()

        scaled_continuous = pd.DataFrame(
            scaler.transform(df[continuous_features]),
            columns=continuous_features,
            index=df.index
        )

        # Keep binary features unscaled
        binary_df = df[binary_features]

        # Keep one-hot encoded features unscaled
        one_hot_df = df[one_hot_columns]

        # Reconstruct dataframe
        scaled_df = pd.concat([scaled_continuous, binary_df, one_hot_df], axis=1)

        # Maintain original column order
        scaled_df = scaled_df[df.columns]

        return scaled_df

    X_train_scaled = scale_df(X_train)
    X_val_scaled = scale_df(X_val)
    X_test_scaled = scale_df(X_test)

    print("Metadata features scaled successfully (train-fit scaler)")
    print(f"Train shape: {X_train_scaled.shape}")
    print(f"Val shape:   {X_val_scaled.shape}")
    print(f"Test shape:  {X_test_scaled.shape}")

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

In [ ]:
# scale metadata features
X_train_scaled, X_val_scaled, X_test_scaled, scaler_meta = scale_metadata_features(
    X_train_meta_encoded,
    X_val_meta_encoded,
    X_test_meta_encoded,
    binary_features,
    one_hot_columns
)
print("Metadata features scaled successfully (train-fit scaler)")
print(f"Train shape: {X_train_scaled.shape}")
print(f"Val shape:   {X_val_scaled.shape}")
print(f"Test shape:  {X_test_scaled.shape}")

Binary features (6): ['creator_has_admin_message_badge', 'creator_ppo_has_action', 'prelaunch_activated', 'video', 'has_reward_images', 'has_restricted_shipping']
Continuous features (22): ['average_number_of_words_per_reward', 'blurb_char_length', 'campaign_duration_days', 'creator_backing_action_count', 'days_between_created_launched', 'goal_converted_USD_log', 'highest_pledge_level_usd', 'is_local', 'launched_day', 'launched_day_of_week', 'launched_hour', 'launched_month', 'lowest_pledge_level_usd', 'num_animated_content', 'num_reward_tiers', 'num_static_images', 'ratio_text_to_total_assets', 'rewards_char_length', 'risks_char_length', 'story_char_length', 'total_media_assets', 'total_text_length']

Metadata features scaled successfully (train-fit scaler).
Train shape: (28210, 91)
Val shape:   (6045, 91)
Test shape:  (6046, 91)
Metadata features scaled successfully (train-fit scaler).
Train shape: (28210, 91)
Val shape:   (6045, 91)
Test shape:  (6046, 91)


In [24]:
# save the metadata
np.save(os.path.join(save_dir, "X_train_scaled.npy"), X_train_scaled.values)
np.save(os.path.join(save_dir, "X_val_scaled.npy"), X_val_scaled.values)
np.save(os.path.join(save_dir, "X_test_scaled.npy"), X_test_scaled.values)

# save labels
np.save(os.path.join(save_dir, "y_train_meta.npy"), y_train_meta.values)
np.save(os.path.join(save_dir, "y_val_meta.npy"), y_val_meta.values)
np.save(os.path.join(save_dir, "y_test_meta.npy"), y_test_meta.values)

---
Text features

In [25]:
# convert scaled metadata to sparse for stacking with sparse text
X_train_meta_scaled_sparse = csr_matrix(X_train_scaled)
X_val_meta_scaled_sparse = csr_matrix(X_val_scaled)
X_test_meta_scaled_sparse = csr_matrix(X_test_scaled)

metadata_feature_names = X_train_scaled.columns.tolist()

In [26]:
# convert scaled metadata to dense for stacking with dense text features
X_train_meta_dense = X_train_scaled.values
X_val_meta_dense   = X_val_scaled.values
X_test_meta_dense  = X_test_scaled.values

In [27]:
# Sparse metadata (for stacking with sparse text)
sparse.save_npz(os.path.join(save_dir, "X_train_meta_scaled_sparse.npz"), X_train_meta_scaled_sparse)
sparse.save_npz(os.path.join(save_dir, "X_val_meta_scaled_sparse.npz"),   X_val_meta_scaled_sparse)
sparse.save_npz(os.path.join(save_dir, "X_test_meta_scaled_sparse.npz"),  X_test_meta_scaled_sparse)

# Dense metadata (for stacking with dense embeddings like BERT)
np.save(os.path.join(save_dir, "X_train_meta_dense.npy"), X_train_meta_dense)
np.save(os.path.join(save_dir, "X_val_meta_dense.npy"),   X_val_meta_dense)
np.save(os.path.join(save_dir, "X_test_meta_dense.npy"),  X_test_meta_dense)

# Metadata feature names
joblib.dump(metadata_feature_names, os.path.join(save_dir, "metadata_feature_names.pkl"))

['saved_features_new/metadata_feature_names.pkl']

In [ ]:
# Text features

# TF-IDF
tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,2), min_df=5, max_df=0.9, sublinear_tf=True, stop_words='english')
X_text_train_tfidf = tfidf.fit_transform(X_train_nlp)
X_text_val_tfidf   = tfidf.transform(X_val_nlp)
X_text_test_tfidf  = tfidf.transform(X_test_nlp)

text_feature_names_tfidf = tfidf.get_feature_names_out().tolist()

# BoW
bow = CountVectorizer(max_features=5000, ngram_range=(1,2))
X_text_train_bow = bow.fit_transform(X_train_nlp)
X_text_val_bow   = bow.transform(X_val_nlp)
X_text_test_bow  = bow.transform(X_test_nlp)

text_feature_names_bow = bow.get_feature_names_out().tolist()

# BERT embeddings
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
X_text_train_bert = bert_model.encode(X_train_llm.tolist(), show_progress_bar=True)
X_text_val_bert   = bert_model.encode(X_val_llm.tolist(), show_progress_bar=True)
X_text_test_bert  = bert_model.encode(X_test_llm.tolist(), show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3757.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 189/189 [01:38<00:00,  1.93it/s]


In [ ]:
# save text features
# TF-IDF
sparse.save_npz(os.path.join(save_dir, "X_text_train_tfidf.npz"), X_text_train_tfidf)
sparse.save_npz(os.path.join(save_dir, "X_text_val_tfidf.npz"),   X_text_val_tfidf)
sparse.save_npz(os.path.join(save_dir, "X_text_test_tfidf.npz"),  X_text_test_tfidf)

joblib.dump(tfidf, os.path.join(save_dir, "tfidf_vectorizer.pkl"))
joblib.dump(text_feature_names_tfidf, os.path.join(save_dir, "tfidf_feature_names.pkl"))

# BoW
sparse.save_npz(os.path.join(save_dir, "X_text_train_bow.npz"), X_text_train_bow)
sparse.save_npz(os.path.join(save_dir, "X_text_val_bow.npz"),   X_text_val_bow)
sparse.save_npz(os.path.join(save_dir, "X_text_test_bow.npz"),  X_text_test_bow)

joblib.dump(bow, os.path.join(save_dir, "bow_vectorizer.pkl"))
joblib.dump(text_feature_names_bow, os.path.join(save_dir, "bow_feature_names.pkl"))

# BERT embeddings (dense)
np.save(os.path.join(save_dir, "X_text_train_bert.npy"), X_text_train_bert)
np.save(os.path.join(save_dir, "X_text_val_bert.npy"),   X_text_val_bert)
np.save(os.path.join(save_dir, "X_text_test_bert.npy"),  X_text_test_bert)

In [30]:
# combine metadata and text features
def combine_sparse(meta_sparse, text_sparse):
    return hstack([meta_sparse, text_sparse])
                
# TF-IDF + metadata
X_train_combined_tfidf = combine_sparse(X_train_meta_scaled_sparse, X_text_train_tfidf)
X_val_combined_tfidf   = combine_sparse(X_val_meta_scaled_sparse, X_text_val_tfidf)
X_test_combined_tfidf  = combine_sparse(X_test_meta_scaled_sparse, X_text_test_tfidf)

combined_feature_names_tfidf = metadata_feature_names + text_feature_names_tfidf

# BoW + metadata
X_train_combined_bow = combine_sparse(X_train_meta_scaled_sparse, X_text_train_bow)
X_val_combined_bow   = combine_sparse(X_val_meta_scaled_sparse, X_text_val_bow)
X_test_combined_bow  = combine_sparse(X_test_meta_scaled_sparse, X_text_test_bow)

combined_feature_names_bow = metadata_feature_names + text_feature_names_bow

# BERT + metadata (dense)
X_train_combined_bert = np.hstack([X_train_meta_dense, X_text_train_bert])
X_val_combined_bert   = np.hstack([X_val_meta_dense, X_text_val_bert])
X_test_combined_bert  = np.hstack([X_test_meta_dense, X_text_test_bert])

combined_feature_names_bert = metadata_feature_names + [f'bert_dim_{i}' for i in range(X_text_train_bert.shape[1])]

print(f"Combined TF-IDF + metadata shape: {X_train_combined_tfidf.shape}")
print(f"Combined BoW + metadata shape: {X_train_combined_bow.shape}") 
print(f"Combined BERT + metadata shape: {X_train_combined_bert.shape}")

Combined TF-IDF + metadata shape: (28210, 8091)
Combined BoW + metadata shape: (28210, 5091)
Combined BERT + metadata shape: (28210, 475)


In [31]:
# save the combined text+meta features
# save combined BoW + metadata
save_npz(os.path.join(save_dir, "X_train_combined_bow.npz"), X_train_combined_bow)
save_npz(os.path.join(save_dir, "X_val_combined_bow.npz"), X_val_combined_bow)
save_npz(os.path.join(save_dir, "X_test_combined_bow.npz"), X_test_combined_bow)

# save combined TF-IDF + metadata
save_npz(os.path.join(save_dir, "X_train_combined_tfidf.npz"), X_train_combined_tfidf)
save_npz(os.path.join(save_dir, "X_val_combined_tfidf.npz"), X_val_combined_tfidf)
save_npz(os.path.join(save_dir, "X_test_combined_tfidf.npz"), X_test_combined_tfidf)

# save non fine-tuned BERT embeddings
np.save(os.path.join(save_dir, "X_train_combined_bert.npy"), X_train_combined_bert)
np.save(os.path.join(save_dir, "X_val_combined_bert.npy"), X_val_combined_bert)
np.save(os.path.join(save_dir, "X_test_combined_bert.npy"), X_test_combined_bert)

In [32]:
# save feature names
with open(os.path.join(save_dir, "combined_feature_names_tfidf.json"), "w") as f:
    json.dump(combined_feature_names_tfidf, f)

with open(os.path.join(save_dir, "combined_feature_names_bow.json"), "w") as f:
    json.dump(combined_feature_names_bow, f)

with open(os.path.join(save_dir, "combined_feature_names_bert.json"), "w") as f:
    json.dump(combined_feature_names_bert, f)

Structured features

Separate Vectors

In [ ]:
# Directory for saving structured features
parent_dir = "saved_features_new"
save_dir2 = os.path.join(parent_dir, "saved_features_structured")
os.makedirs(save_dir2, exist_ok=True)

# Structured columns
structured_cols_nlp = {
    "name": "name_nlp",
    "blurb": "blurb_nlp",
    "story": "story_html_text_nlp",
    "risks": "risks_html_text_nlp",
    "rewards": "rewards_html_text_nlp"
}

structured_cols_llm = {
    "name": "name_llm",
    "blurb": "blurb_llm",
    "story": "story_html_text_llm",
    "risks": "risks_html_text_llm",
    "rewards": "rewards_html_text_llm"
}

# Structured TF-IDF + metadata
tfidf_vectors = []
tfidf_feature_names_structured = []

# For separate fields (deep models)
tfidf_vectors_only = {}
tfidf_feature_names_only = {}

# Process each structured text field separately and save both combined and separate versions
for section, col in structured_cols_nlp.items():
    vec = TfidfVectorizer(
        max_features=2000, ngram_range=(1,2),
        min_df=3, max_df=0.85,
        sublinear_tf=True, norm='l2',
        stop_words='english'
    )
    X_tr = vec.fit_transform(preprocessed_df.loc[splits["train_idx"], col].fillna(''))
    X_val_ = vec.transform(preprocessed_df.loc[splits["val_idx"], col].fillna(''))
    X_te = vec.transform(preprocessed_df.loc[splits["test_idx"], col].fillna(''))

    tfidf_vectors.append((X_tr, X_val_, X_te))
    tfidf_feature_names_structured.extend([f"{section}_tfidf_{f}" for f in vec.get_feature_names_out()])

    # Separate per field
    tfidf_vectors_only[section] = (X_tr, X_val_, X_te)
    tfidf_feature_names_only[section] = [f"{section}_tfidf_{f}" for f in vec.get_feature_names_out()]

    # Save separate fields
    sparse.save_npz(os.path.join(save_dir2, f"X_train_tfidf_{section}.npz"), X_tr)
    sparse.save_npz(os.path.join(save_dir2, f"X_val_tfidf_{section}.npz"), X_val_)
    sparse.save_npz(os.path.join(save_dir2, f"X_test_tfidf_{section}.npz"), X_te)
    joblib.dump(tfidf_feature_names_only[section], os.path.join(save_dir2, f"feature_names_tfidf_{section}.pkl"))

# Combine all structured TF-IDF fields with metadata
X_train_tfidf_combined_structured = hstack([X_train_meta_scaled_sparse] + [v[0] for v in tfidf_vectors])
X_val_tfidf_combined_structured   = hstack([X_val_meta_scaled_sparse] + [v[1] for v in tfidf_vectors])
X_test_tfidf_combined_structured  = hstack([X_test_meta_scaled_sparse] + [v[2] for v in tfidf_vectors])

# Combined feature names for structured TF-IDF + metadata
combined_feature_names_tfidf_structured = metadata_feature_names + tfidf_feature_names_structured

# Save combined structured TF-IDF + metadata features
sparse.save_npz(os.path.join(save_dir2, "X_train_tfidf_combined_structured.npz"), X_train_tfidf_combined_structured)
sparse.save_npz(os.path.join(save_dir2, "X_val_tfidf_combined_structured.npz"), X_val_tfidf_combined_structured)
sparse.save_npz(os.path.join(save_dir2, "X_test_tfidf_combined_structured.npz"), X_test_tfidf_combined_structured)
joblib.dump(combined_feature_names_tfidf_structured, os.path.join(save_dir2, "combined_feature_names_tfidf_structured.pkl"))

# Structured BoW + metadata
bow_vectors = []
bow_feature_names_structured = []

# For separate fields
bow_vectors_only = {}
bow_feature_names_only = {}

# Process each structured text field separately and save both combined and separate versions
for section, col in structured_cols_nlp.items():
    vec = CountVectorizer(max_features=2000, ngram_range=(1,2))
    X_tr = vec.fit_transform(preprocessed_df.loc[splits["train_idx"], col].fillna(''))
    X_val_ = vec.transform(preprocessed_df.loc[splits["val_idx"], col].fillna(''))
    X_te = vec.transform(preprocessed_df.loc[splits["test_idx"], col].fillna(''))

    bow_vectors.append((X_tr, X_val_, X_te))
    bow_feature_names_structured.extend([f"{section}_bow_{f}" for f in vec.get_feature_names_out()])

     # Separate per field
    bow_vectors_only[section] = (X_tr, X_val_, X_te)
    bow_feature_names_only[section] = [f"{section}_bow_{f}" for f in vec.get_feature_names_out()]

    # Save separate fields
    sparse.save_npz(os.path.join(save_dir2, f"X_train_bow_{section}.npz"), X_tr)
    sparse.save_npz(os.path.join(save_dir2, f"X_val_bow_{section}.npz"), X_val_)
    sparse.save_npz(os.path.join(save_dir2, f"X_test_bow_{section}.npz"), X_te)
    joblib.dump(bow_feature_names_only[section], os.path.join(save_dir2, f"feature_names_bow_{section}.pkl"))

# Combine all structured BoW fields with metadata
X_train_bow_combined_structured = hstack([X_train_meta_scaled_sparse] + [v[0] for v in bow_vectors])
X_val_bow_combined_structured   = hstack([X_val_meta_scaled_sparse] + [v[1] for v in bow_vectors])
X_test_bow_combined_structured  = hstack([X_test_meta_scaled_sparse] + [v[2] for v in bow_vectors])

# Combined feature names for structured BoW + metadata
combined_feature_names_bow_structured = metadata_feature_names + bow_feature_names_structured

# Save combined structured BoW + metadata features
sparse.save_npz(os.path.join(save_dir2, "X_train_combined_bow_structured.npz"), X_train_bow_combined_structured)
sparse.save_npz(os.path.join(save_dir2, "X_val_combined_bow_structured.npz"), X_val_bow_combined_structured)
sparse.save_npz(os.path.join(save_dir2, "X_test_combined_bow_structured.npz"), X_test_bow_combined_structured)
joblib.dump(combined_feature_names_bow_structured, os.path.join(save_dir2, "combined_feature_names_bow_structured.pkl"))

# Structured BERT + metadata
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
bert_embeddings_train, bert_embeddings_val, bert_embeddings_test = [], [], []
bert_feature_names_structured = []

# Separate per field
bert_embeddings_only = {}
bert_feature_names_only = {}

# Process each structured text field separately and save both combined and separate versions
for section, col in structured_cols_llm.items():
    X_tr = bert_model.encode(preprocessed_df.loc[splits["train_idx"], col].fillna('').tolist(), show_progress_bar=True)
    X_val_ = bert_model.encode(preprocessed_df.loc[splits["val_idx"], col].fillna('').tolist(), show_progress_bar=True)
    X_te = bert_model.encode(preprocessed_df.loc[splits["test_idx"], col].fillna('').tolist(), show_progress_bar=True)

    bert_embeddings_train.append(X_tr)
    bert_embeddings_val.append(X_val_)
    bert_embeddings_test.append(X_te)

    # Feature names with section prefix
    n_dims = X_tr.shape[1]
    bert_feature_names_structured += [f"{section}_bert_{i}" for i in range(n_dims)]

    # Separate per field
    bert_embeddings_only[section] = (X_tr, X_val_, X_te)
    bert_feature_names_only[section] = [f"{section}_bert_{i}" for i in range(n_dims)]

    # Save separate fields 
    np.save(os.path.join(save_dir2, f"X_train_bert_{section}.npy"), X_tr)
    np.save(os.path.join(save_dir2, f"X_val_bert_{section}.npy"), X_val_)
    np.save(os.path.join(save_dir2, f"X_test_bert_{section}.npy"), X_te)
    joblib.dump(bert_feature_names_only[section], os.path.join(save_dir2, f"feature_names_bert_{section}.pkl"))

# Combine all structured BERT fields with metadata
X_train_bert_combined_structured = np.hstack([X_train_meta_dense] + bert_embeddings_train)
X_val_bert_combined_structured   = np.hstack([X_val_meta_dense] + bert_embeddings_val)
X_test_bert_combined_structured  = np.hstack([X_test_meta_dense] + bert_embeddings_test)

# Combined feature names for structured BERT + metadata
combined_feature_names_bert_structured = metadata_feature_names + bert_feature_names_structured

# Save combined structured BERT + metadata features
np.save(os.path.join(save_dir2, "X_train_combined_bert_structured.npy"), X_train_bert_combined_structured)
np.save(os.path.join(save_dir2, "X_val_combined_bert_structured.npy"), X_val_bert_combined_structured)
np.save(os.path.join(save_dir2, "X_test_combined_bert_structured.npy"), X_test_bert_combined_structured)
joblib.dump(combined_feature_names_bert_structured, os.path.join(save_dir2, "combined_feature_names_bert_structured.pkl"))

print(f"Saved all combined structured features with semantic prefixes to {save_dir2}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2662.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 189/189 [01:15<00:00,  2.49it/s]


Saved all combined structured features with semantic prefixes to saved_features_new/saved_features_structured


---
Fine-tuned embeddings

In [ ]:
# BERT model for fine-tuning
model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Load tokenizer and model for fine-tuning
tokenizer = AutoTokenizer.from_pretrained(model_name)

# custom PyTorch model that uses BERT embeddings with a projection head and classifier on top
class BertEmbeddingClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        # Freeze encoder
        for param in self.encoder.parameters():
            param.requires_grad = False

        # Fine-tune last 4 transformer layers
        for layer in self.encoder.encoder.layer[-4:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Get hidden size from the encoder config
        hidden = self.encoder.config.hidden_size

        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Classifier head
        self.classifier = nn.Linear(256, 2)

    # Forward method that takes input_ids and attention_mask, extracts the last 4 layers, averages them, applies mean pooling, and passes through projection and classifier heads
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True
        )

        # Take last 4 transformer layers
        hidden_states = outputs.hidden_states[-4:]
        # Stack → shape = [4, batch, seq, hidden]
        stacked = torch.stack(hidden_states)
        # Average across layers → [batch, seq, hidden]
        hidden = torch.mean(stacked, dim=0)
        # Mean pooling over tokens (ignore PAD tokens)
        mask = attention_mask.unsqueeze(-1)

        # Avoid division by zero by using clamp on the sum of the mask
        pooled = torch.sum(hidden * mask, dim=1) / torch.clamp(
        mask.sum(dim=1),
        min=1e-9
        )

        # Projection head
        projected = self.projection(pooled)
        logits = self.classifier(projected)
        return logits
        
    # Save method to save encoder and projection head separately
    def save_pretrained(self, save_directory):
        # Save encoder and projection head separately
        self.encoder.save_pretrained(save_directory)
        torch.save(
            self.projection.state_dict(),
            f"{save_directory}/projection_head.pt"
        )
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertEmbeddingClassifier(model_name).to(device)

In [ ]:
# Custom Dataset class for text data
class TextDataset(Dataset):
    # Initialize with texts and labels, tokenize the texts, and store encodings and labels
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=384,
        )
        self.labels = labels.values

    # Get item by index, return a dictionary of input_ids, attention_mask, and labels as tensors
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    # Return the length of the dataset
    def __len__(self):
        return len(self.labels)

In [ ]:
# Create datasets and dataloaders for the LLM text data
train_dataset = TextDataset(X_train_llm, y_train_llm)
val_dataset   = TextDataset(X_val_llm, y_val_llm)

# Create dataloaders with batch size 16 and shuffle the training data
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)

In [ ]:
# Print the number of samples in each split for the LLM setting
print(len(X_train_llm))
print(len(y_train_llm))
print(len(X_val_llm))
print(len(y_val_llm))

In [ ]:
# Function to evaluate the model on a given dataloader, compute average loss and AUC-ROC score
def evaluate(model, dataloader):
    model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            # Move inputs and labels to the device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Get logits from the model
            logits = model(input_ids, attention_mask)

            # Compute cross-entropy loss with label smoothing
            loss = nn.CrossEntropyLoss(
                label_smoothing=0.1
            )(logits, labels)

            # Accumulate total loss
            total_loss += loss.item()

            # Compute probabilities for the positive class (index 1)
            probs = torch.softmax(logits, dim=1)[:, 1]

            # Accumulate probabilities and labels for AUC-ROC calculation
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute average loss and AUC-ROC score
    avg_loss = total_loss / len(dataloader)
    auc = roc_auc_score(all_labels, all_probs)

    return avg_loss, auc

In [ ]:
# Configuration
optimizer = AdamW(model.parameters(), lr=3e-5)
epochs = 15
patience = 3   
best_val_auc = 0.0
epochs_no_improve = 0
best_model_state = None

print(f"Starting training on {device}...")

for epoch in range(epochs):
    # Training phase
    model.train()
    total_train_loss = 0
    
    # Progress bar for the training loader
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for batch in train_loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        
        # Using label smoothing
        loss = nn.CrossEntropyLoss(label_smoothing=0.1)(logits, labels)       
        
        loss.backward()
        # Clip gradients to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # Accumulate training loss and update progress bar
        total_train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation phase
    val_loss, val_auc = evaluate(model, val_loader)

    print(f"\nSummary - Epoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val AUC:    {val_auc:.4f}")

    # Early stopping based on AUC improvement
    if val_auc > (best_val_auc + 1e-4):
        best_val_auc = val_auc
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        print(f"New Best Val AUC: {val_auc:.4f} — Model Saved")
    else:
        epochs_no_improve += 1
        print(f"No significant AUC improvement for {epochs_no_improve} epoch(s)")

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at Epoch {epoch+1}")
            break

# Restore best weights
if best_model_state:
    model.load_state_dict(best_model_state)
    print("Best model weights restored based on Val AUC.")

In [ ]:
# Save the best model
model.load_state_dict(best_model_state)
print("Best model restored.")

In [ ]:
# Load base model with fine-tuned weights
base_model = model.encoder
base_model.eval()
model.projection.eval()

In [ ]:
# Function to extract mean-pooled embeddings from the fine-tuned BERT model for a given series of texts, processing in batches
def extract_embeddings_mean_pooling(text_series, batch_size=32):

    embeddings = []

    base_model.eval()
    model.eval()

    # Process texts in batches to manage memory usage
    for i in range(0, len(text_series), batch_size):

        batch_texts = text_series.iloc[i:i+batch_size].fillna('').tolist()

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=384,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():

            outputs = base_model(
                **inputs,
                output_hidden_states=True
            )
            # Take last 4 transformer layers
            hidden_states = outputs.hidden_states[-4:]
            # Stack → shape = [4, batch, seq, hidden]
            stacked = torch.stack(hidden_states)
            # Average across layers → [batch, seq, hidden]
            hidden = torch.mean(stacked, dim=0)
            # Mean pooling over tokens (ignore PAD tokens)
            mask = inputs["attention_mask"].unsqueeze(-1)
            # Avoid division by zero by using clamp on the sum of the mask
            pooled = torch.sum(hidden * mask, dim=1) / torch.clamp(
                mask.sum(dim=1),
                min=1e-9
            )

            # Projected embeddings from the projection head
            with torch.no_grad():
                projected = model.projection(pooled)

        embeddings.append(projected.cpu())

    embeddings = torch.cat(embeddings).numpy()

    return embeddings

In [ ]:
# Extract fine-tuned BERT embeddings for the LLM text splits using mean pooling
X_text_train_bert_ft = extract_embeddings_mean_pooling(X_train_llm)
X_text_val_bert_ft = extract_embeddings_mean_pooling(X_val_llm)
X_text_test_bert_ft = extract_embeddings_mean_pooling(X_test_llm)

In [ ]:
# Print shapes of the original and fine-tuned BERT embeddings to verify
print(X_text_train_bert.shape)

print("Train embeddings:")
print(X_text_train_bert_ft.shape)

print("Validation embeddings:")
print(X_text_val_bert_ft.shape)

print("Test embeddings:")
print(X_text_test_bert_ft.shape)

In [ ]:
# Save the fine-tuned BERT embeddings for the LLM text splits
np.save(os.path.join(save_dir, "X_train_bert_ft.npy"), X_text_train_bert_ft)
np.save(os.path.join(save_dir, "X_val_bert_ft.npy"), X_text_val_bert_ft)
np.save(os.path.join(save_dir, "X_test_bert_ft.npy"), X_text_test_bert_ft)

In [ ]:
# Save the fine-tuned BERT model and tokenizer for future use
model.save_pretrained("./bert_finetuned_model2")
tokenizer.save_pretrained("./bert_finetuned_model2")

# Save projection head weights
torch.save(
    model.projection.state_dict(),
    "./bert_finetuned_model2/projection_head.pt"
)

In [ ]:
# Fine-tuned BERT (Raw + Projection) + Metadata
X_train_combined_bert_ft = np.hstack([X_train_meta_dense, X_text_train_bert_ft])

X_val_combined_bert_ft = np.hstack([X_val_meta_dense, X_text_val_bert_ft])

X_test_combined_bert_ft = np.hstack([X_test_meta_dense, X_text_test_bert_ft])

combined_feature_names_bert_ft = metadata_feature_names + [
    f'bert_ft_dim_{i}' for i in range(X_text_train_bert_ft.shape[1])
]

print(f"Fine-tuned BERT + metadata shape: {X_train_combined_bert_ft.shape}")

In [ ]:
# BERT + metadata (fine-tuned embeddings)
np.save(os.path.join(save_dir, "X_train_combined_bert_ft.npy"), X_train_combined_bert_ft)
np.save(os.path.join(save_dir, "X_val_combined_bert_ft.npy"), X_val_combined_bert_ft)
np.save(os.path.join(save_dir, "X_test_combined_bert_ft.npy"), X_test_combined_bert_ft)

In [ ]:
# Feature names for fine-tuned BERT + metadata
with open(os.path.join(save_dir, "combined_feature_names_bert_ft.json"), "w") as f:
    json.dump(combined_feature_names_bert_ft, f)

Fine-tuned structured embeddings

In [ ]:
# extract and save fine-tuned BERT embeddings for each structured text field separately, both with metadata and on their own
bert_embeddings_val = []
bert_embeddings_test = []

bert_feature_names_structured = []

bert_embeddings_only = {}
bert_feature_names_only = {}

# Process each structured text field separately and save both combined and separate versions
for section, col in structured_cols_llm.items():

    print(f"Extracting BERT embeddings for {section}...")

    train_embeddings = extract_embeddings_mean_pooling(preprocessed_df.loc[splits["train_idx"], col].fillna(''))

    val_embeddings = extract_embeddings_mean_pooling(preprocessed_df.loc[splits["val_idx"], col].fillna(''))

    test_embeddings = extract_embeddings_mean_pooling(preprocessed_df.loc[splits["test_idx"], col].fillna(''))

    # only keep projection embeddings
    bert_embeddings_train.append(train_embeddings)
    bert_embeddings_val.append(val_embeddings)
    bert_embeddings_test.append(test_embeddings)

    n_dims = train_embeddings.shape[1]

    bert_feature_names_structured += [
        f"{section}_bert_ft_{i}" for i in range(n_dims)
    ]

    bert_embeddings_only[section] = (train_embeddings, val_embeddings, test_embeddings)

    bert_feature_names_only[section] = [
        f"{section}_bert_ft_{i}" for i in range(n_dims)
    ]

    # Save per section fine-tuned BERT embeddings
    np.save(os.path.join(save_dir2, f"X_train_bert_ft_{section}.npy"), train_embeddings)
    np.save(os.path.join(save_dir2, f"X_val_bert_ft_{section}.npy"), val_embeddings)
    np.save(os.path.join(save_dir2, f"X_test_bert_ft_{section}.npy"), test_embeddings)

    joblib.dump(
        bert_feature_names_only[section],
        os.path.join(save_dir2, f"feature_names_bert_ft_{section}.pkl")
    )

In [ ]:
# Combine all structured fine-tuned BERT fields with metadata
X_train_bert_ft_structured = np.hstack(bert_embeddings_train)
X_val_bert_ft_structured   = np.hstack(bert_embeddings_val)
X_test_bert_ft_structured  = np.hstack(bert_embeddings_test)

feature_names_bert_ft_structured = bert_feature_names_structured

In [ ]:
# Save combined structured fine-tuned BERT + metadata features
np.save(os.path.join(save_dir2, "X_train_bert_ft_structured.npy"), X_train_bert_ft_structured)
np.save(os.path.join(save_dir2, "X_val_bert_ft_structured.npy"), X_val_bert_ft_structured)
np.save(os.path.join(save_dir2, "X_test_bert_ft_structured.npy"), X_test_bert_ft_structured)

joblib.dump(
    feature_names_bert_ft_structured,
    os.path.join(save_dir2, "feature_names_bert_ft_structured.pkl")
)

In [ ]:
# create combined versions of the fine-tuned BERT embeddings with metadata for the structured fields
X_train_bert_combined_ft_structured = np.hstack([X_train_meta_dense] + bert_embeddings_train)
X_val_bert_combined_ft_structured   = np.hstack([X_val_meta_dense] + bert_embeddings_val)
X_test_bert_combined_ft_structured  = np.hstack([X_test_meta_dense] + bert_embeddings_test)

combined_feature_names_bert_ft_structured = (
    metadata_feature_names + bert_feature_names_structured
)

In [ ]:
# Save the combined fine-tuned BERT + metadata features for the structured fields
np.save(os.path.join(save_dir2, "X_train_combined_bert_ft_structured.npy"), X_train_bert_combined_ft_structured)
np.save(os.path.join(save_dir2, "X_val_combined_bert_ft_structured.npy"), X_val_bert_combined_ft_structured)
np.save(os.path.join(save_dir2, "X_test_combined_bert_ft_structured.npy"), X_test_bert_combined_ft_structured)

joblib.dump(
    combined_feature_names_bert_ft_structured,
    os.path.join(save_dir2, "combined_feature_names_bert_ft_structured.pkl")
)

print(f"Saved fine-tuned structured BERT features to {save_dir2}")

Meta + Text + Images

In [ ]:
# Load image embeddings memmap
RESNET_DIM = 2048
N = len(preprocessed_df)

# Create a memory-mapped array for the image embeddings
image_embeddings = np.memmap(
    "./image_embeddings_final_new/resnet_embeddings.dat",
    dtype="float32",
    mode="r",
    shape=(N, RESNET_DIM)
)

image_embeddings = np.array(image_embeddings)   # convert to normal array for ML training

In [ ]:
# Print the shape of the loaded image embeddings
print("Embedding matrix shape:", image_embeddings.shape)
assert image_embeddings.shape[0] == len(preprocessed_df)

Embedding matrix shape: (40301, 2048)


In [ ]:
# Extract image embeddings for each split using the train/val/test indices
X_train_img = image_embeddings[train_idx]
X_val_img   = image_embeddings[val_idx]
X_test_img  = image_embeddings[test_idx]

In [ ]:
# Save the image embeddings for each split as .npy files
np.save(os.path.join(save_dir, "X_train_img.npy"), X_train_img)
np.save(os.path.join(save_dir, "X_val_img.npy"),   X_val_img)
np.save(os.path.join(save_dir, "X_test_img.npy"),  X_test_img)

In [ ]:
# combine the dense metadata, BERT embeddings, and image embeddings into a single multimodal feature set for each split
X_train_multimodal = np.hstack([X_train_meta_dense, X_text_train_bert, X_train_img])
X_val_multimodal = np.hstack([X_val_meta_dense, X_text_val_bert, X_val_img])
X_test_multimodal = np.hstack([X_test_meta_dense, X_text_test_bert, X_test_img])

meta_names = metadata_feature_names
bert_names = [f"bert_dim_{i}" for i in range(X_text_train_bert.shape[1])]
img_names  = [f"img_dim_{i}" for i in range(X_train_img.shape[1])]

multimodal_feature_names = meta_names + bert_names + img_names

In [ ]:
# Combine the dense metadata, fine-tuned BERT embeddings, and image embeddings into a single multimodal feature set for each split 
X_train_multimodal_ft = np.hstack([X_train_meta_dense, X_text_train_bert_ft, X_train_img])
X_val_multimodal_ft = np.hstack([X_val_meta_dense, X_text_val_bert_ft, X_val_img])
X_test_multimodal_ft = np.hstack([X_test_meta_dense, X_text_test_bert_ft, X_test_img])

bert_ft_names = [f"bert_ft_dim_{i}" for i in range(X_text_train_bert_ft.shape[1])]

multimodal_ft_feature_names = meta_names + bert_ft_names + img_names

X_train_multimodal_ft_structured = np.hstack([X_train_meta_dense, X_train_bert_ft_structured, X_train_img])
X_val_multimodal_ft_structured = np.hstack([X_val_meta_dense, X_val_bert_ft_structured, X_val_img])
X_test_multimodal_ft_structured = np.hstack([X_test_meta_dense, X_test_bert_ft_structured, X_test_img])

# Feature names
multimodal_ft_structured_feature_names = (metadata_feature_names + bert_feature_names_structured + img_names)

In [ ]:
# Multimodal (static BERT)
np.save(os.path.join(save_dir, "X_train_multimodal.npy"), X_train_multimodal)
np.save(os.path.join(save_dir, "X_val_multimodal.npy"),   X_val_multimodal)
np.save(os.path.join(save_dir, "X_test_multimodal.npy"),  X_test_multimodal)

# Multimodal (fine-tuned BERT)
np.save(os.path.join(save_dir, "X_train_multimodal_ft.npy"), X_train_multimodal_ft)
np.save(os.path.join(save_dir, "X_val_multimodal_ft.npy"),   X_val_multimodal_ft)
np.save(os.path.join(save_dir, "X_test_multimodal_ft.npy"),  X_test_multimodal_ft)

# Multimodal Structured (fine-tuned BERT)
np.save(os.path.join(save_dir, "X_train_multimodal_ft_structured.npy"), X_train_multimodal_ft_structured)
np.save(os.path.join(save_dir, "X_val_multimodal_ft_structured.npy"),   X_val_multimodal_ft_structured)
np.save(os.path.join(save_dir, "X_test_multimodal_ft_structured.npy"),  X_test_multimodal_ft_structured)

In [ ]:
# Save feature names for the multimodal feature sets
joblib.dump(multimodal_feature_names,
            os.path.join(save_dir, "multimodal_feature_names.pkl"))

joblib.dump(multimodal_ft_feature_names,
            os.path.join(save_dir, "multimodal_ft_feature_names.pkl"))

joblib.dump(multimodal_ft_structured_feature_names,
            os.path.join(save_dir, "multimodal_ft_feature_names_structured.pkl"))